# 🧸 Experiment 01. Accuracy vs. Cohen's 𝜅 🧸 

This experiment examines why observed agreement and Cohen's κ change differently.

Using the original baseline confusion matrix (n = 300), I explore how changes in observed agreement ($P_o$) and expected agreement ($P_e$) affect κ.

## Contents

1. **Question**
2. **Manipulation**
   - A. Hold observed agreement constant
   - B. Hold expected agreement constant
   - Predictions
3. **Baseline Verification**
4. **Experimental Execution**
   - 4.1. Experiment A1 — Decrease $P_e$
   - 4.2. Experiment A2 — Increase $P_e$
   - 4.3. Experiment B — Change $P_o$
5. **Observations**
6. **Remaining Questions**

## 1. Question

Why do observed agreement and Cohen's κ change differently?

In the original study, the exclusion-rule replacement condition showed the following changes from baseline:

| Measure | Baseline | Exclusion-rule replacement | Change |
|---|---:|---:|---:|
| Observed agreement ($P_o$) | 0.7867 | 0.6967 | −0.0900 |
| Expected agreement ($P_e$) | 0.4551 | 0.3906 | −0.0645 |
| Cohen's κ | 0.6085 | 0.5023 | −0.1062 |

Observed agreement decreased by 0.0900, while Cohen's κ decreased by 0.1062.

Why are these changes different?

Cohen's κ depends on both observed agreement ($P_o$) and expected agreement ($P_e$).

This experiment examines how changes in each measure affect κ.

## 2. Manipulation

This experiment uses the original baseline confusion matrix (n = 300).

All changes are hypothetical. Each manipulation starts from a separate copy of the original matrix.

### A. Hold Observed Agreement Constant

Change an incorrect LLM label to another incorrect category while keeping the human label unchanged.

**A1. Decrease Expected Agreement**

- Human label: KET
- LLM label: NC → PFR
- Change one prediction.

**A2. Increase Expected Agreement**

- Human label: KET
- LLM label: PFR → NC
- Change one prediction.

Both the original and modified predictions are incorrect.

Therefore, $P_o$ remains constant while the model's category marginals and $P_e$ change.

### B. Hold Expected Agreement Constant

Swap two incorrect LLM predictions.

- Human NC / LLM KET → LLM NC
- Human KET / LLM NC → LLM KET

Both predictions become correct.

The model's category marginals remain unchanged, so $P_e$ stays constant while $P_o$ increases.

### Predictions

- A1: Decreasing $P_e$ while holding $P_o$ constant will increase κ.
- A2: Increasing $P_e$ while holding $P_o$ constant will decrease κ.
- B: Increasing $P_o$ while holding $P_e$ constant will increase κ.

The original confusion matrix and study results remain unchanged.

## 3. Baseline Verification

Before applying the hypothetical manipulations, I reproduce the original baseline statistics using its confusion matrix (n = 300).

The code calculates:

- Observed agreement ($P_o$) from the diagonal values.
- Expected agreement ($P_e$) from the category marginals.
- Cohen's κ from $P_o$ and $P_e$.

The calculated values are compared with the original scoring record.

**Source:** [Main Scoring Record, Section 11](https://github.com/geekinthegit/coding-manual-ablation/blob/main/reports/main-scoring-record-2026-09-24.md)

In [1]:
import numpy as np

# Baseline confusion matrix
# Rows = human labels
# Columns = LLM labels

matrix = np.array([
    [171, 20, 1, 0, 6, 7, 2],
    [7, 22, 1, 1, 4, 4, 2],
    [0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 2, 0, 0, 0],
    [0, 0, 0, 1, 3, 0, 0],
    [1, 3, 0, 0, 0, 35, 3],
    [0, 0, 0, 0, 0, 0, 3]
])

n = matrix.sum()

# Category marginals
human = matrix.sum(axis=1)
model = matrix.sum(axis=0)

# Observed agreement
Po = np.trace(matrix) / n

# Expected agreement
Pe = np.dot(human, model) / n**2

# Cohen's kappa
kappa = (Po - Pe) / (1 - Pe)

assert n == 300

print(f"Observed agreement: {Po:.4f}")
print(f"Expected agreement: {Pe:.4f}")
print(f"Cohen's kappa: {kappa:.4f}")

Observed agreement: 0.7867
Expected agreement: 0.4551
Cohen's kappa: 0.6085


## 4. Experimental Execution

### 4.1. Experiment A1 — Decrease Expected Agreement

**Question:** What happens to Cohen's κ when $P_e$ decreases while $P_o$ remains constant?

**Manipulation**

Change one incorrect LLM prediction for an utterance labeled KET by the human coder.

| Human | LLM | Before | After |
|---|---|---:|---:|
| KET | NC | 7 | 6 |
| KET | PFR | 2 | 3 |

Both NC and PFR are incorrect for a human KET label.

Therefore, $P_o$ remains unchanged, while the model's category marginals change.

**Prediction:** $P_e$ will decrease and Cohen's κ will increase.

The manipulation is hypothetical. The original baseline matrix remains unchanged.

In [2]:
# 4.1. Experiment A1 — Decrease Pe

# Copy the original baseline matrix
A1 = matrix.copy()

# Move one incorrect prediction from NC to PFR
A1[1, 0] -= 1
A1[1, 6] += 1

# Recalculate the statistics
Po_A1 = np.trace(A1) / n
Pe_A1 = np.dot(A1.sum(axis=1), A1.sum(axis=0)) / n**2
kappa_A1 = (Po_A1 - Pe_A1) / (1 - Pe_A1)

# Verify the manipulation
assert A1.sum() == 300
assert np.isclose(Po_A1, Po)
assert Pe_A1 < Pe

# Compare baseline and A1
print("                 Baseline     A1        Change")
print(f"Po               {Po:.4f}       {Po_A1:.4f}    {Po_A1-Po:+.4f}")
print(f"Pe               {Pe:.4f}       {Pe_A1:.4f}    {Pe_A1-Pe:+.4f}")
print(f"Cohen's kappa    {kappa:.4f}       {kappa_A1:.4f}    {kappa_A1-kappa:+.4f}")

                 Baseline     A1        Change
Po               0.7867       0.7867    +0.0000
Pe               0.4551       0.4529    -0.0023
Cohen's kappa    0.6085       0.6101    +0.0016


### 4.2. Experiment A2 — Increase Expected Agreement

**Question:** What happens to Cohen's κ when $P_e$ increases while $P_o$ remains constant?

**Manipulation**

Change one incorrect LLM prediction from PFR to NC for an utterance labeled KET by the human coder.

| Human | LLM | Before | After |
|---|---|---:|---:|
| KET | PFR | 2 | 1 |
| KET | NC | 7 | 8 |

Both PFR and NC are incorrect for a human KET label.

Therefore, $P_o$ remains unchanged, while the model's category marginals change.

**Prediction:** $P_e$ will increase and Cohen's κ will decrease.

This manipulation starts from the original baseline matrix, not the modified A1 matrix.

In [3]:
# 4.2. Experiment A2 — Increase Pe

# Copy the original baseline matrix
A2 = matrix.copy()

# Move one incorrect prediction from PFR to NC
A2[1, 6] -= 1
A2[1, 0] += 1

# Recalculate the statistics
Po_A2 = np.trace(A2) / n
Pe_A2 = np.dot(A2.sum(axis=1), A2.sum(axis=0)) / n**2
kappa_A2 = (Po_A2 - Pe_A2) / (1 - Pe_A2)

# Verify the manipulation
assert A2.sum() == 300
assert np.isclose(Po_A2, Po)
assert Pe_A2 > Pe

# Compare baseline and A2
print("                 Baseline     A2        Change")
print(f"Po               {Po:.4f}       {Po_A2:.4f}    {Po_A2-Po:+.4f}")
print(f"Pe               {Pe:.4f}       {Pe_A2:.4f}    {Pe_A2-Pe:+.4f}")
print(f"Cohen's kappa    {kappa:.4f}       {kappa_A2:.4f}    {kappa_A2-kappa:+.4f}")

                 Baseline     A2        Change
Po               0.7867       0.7867    +0.0000
Pe               0.4551       0.4574    +0.0023
Cohen's kappa    0.6085       0.6068    -0.0016


### 4.3. Experiment B — Increase Observed Agreement

**Question:** What happens to Cohen's κ when $P_o$ increases while $P_e$ remains constant?

**Manipulation**

Swap two incorrect LLM predictions from the original baseline matrix.

| Human | LLM | Before | After |
|---|---|---:|---:|
| NC | KET | 20 | 19 |
| NC | NC | 171 | 172 |
| KET | NC | 7 | 6 |
| KET | KET | 22 | 23 |

Two incorrect predictions become correct.

The total number of NC and KET predictions remains unchanged. Therefore, the model's category marginals and $P_e$ remain constant.

Observed agreement increases by two utterances.

**Prediction:** $P_o$ will increase, $P_e$ will remain constant, and Cohen's κ will increase.

This manipulation starts from the original baseline matrix.

In [4]:
# 4.3. Experiment B — Increase Po

# Copy the original baseline matrix
B = matrix.copy()

# Swap two incorrect LLM predictions
# Human NC: KET -> NC
B[0, 1] -= 1
B[0, 0] += 1

# Human KET: NC -> KET
B[1, 0] -= 1
B[1, 1] += 1

# Recalculate the statistics
Po_B = np.trace(B) / n
Pe_B = np.dot(B.sum(axis=1), B.sum(axis=0)) / n**2
kappa_B = (Po_B - Pe_B) / (1 - Pe_B)

# Verify the manipulation
assert B.sum() == 300
assert np.array_equal(B.sum(axis=0), matrix.sum(axis=0))
assert np.array_equal(B.sum(axis=1), matrix.sum(axis=1))
assert np.isclose(Pe_B, Pe)
assert np.trace(B) == np.trace(matrix) + 2

# Compare baseline and B
print("                 Baseline     B         Change")
print(f"Po               {Po:.4f}       {Po_B:.4f}    {Po_B-Po:+.4f}")
print(f"Pe               {Pe:.4f}       {Pe_B:.4f}    {Pe_B-Pe:+.4f}")
print(f"Cohen's kappa    {kappa:.4f}       {kappa_B:.4f}    {kappa_B-kappa:+.4f}")

                 Baseline     B         Change
Po               0.7867       0.7933    +0.0067
Pe               0.4551       0.4551    +0.0000
Cohen's kappa    0.6085       0.6207    +0.0122


## 5. Observations

All three manipulations started from the original baseline confusion matrix (n = 300).

| Condition | $P_o$ | $P_e$ | Cohen's κ |
|---|---:|---:|---:|
| Baseline | 0.7867 | 0.4551 | 0.6085 |
| A1 — Decrease $P_e$ | 0.7867 | 0.4529 | 0.6101 |
| A2 — Increase $P_e$ | 0.7867 | 0.4574 | 0.6068 |
| B — Increase $P_o$ | 0.7933 | 0.4551 | 0.6207 |

### Experiment A1

Changing one incorrect prediction from NC to PFR decreased $P_e$ by 0.0023.

$P_o$ remained unchanged, while κ increased by 0.0016.

### Experiment A2

Changing one incorrect prediction from PFR to NC increased $P_e$ by 0.0023.

$P_o$ remained unchanged, while κ decreased by 0.0016.

### Experiment B

Changing two incorrect predictions into correct predictions increased $P_o$ by 0.0067.

$P_e$ remained unchanged because the category marginals were preserved.

Cohen's κ increased by 0.0122.

### What I Learned

Cohen's κ depends on both observed agreement and expected agreement.

When $P_o$ is constant, changing $P_e$ changes κ.

When $P_e$ is constant, changing $P_o$ also changes κ, but not by the same amount.

Therefore, changes in observed agreement and Cohen's κ do not have to be equal.

## 6. Remaining Questions

These experiments used hypothetical label manipulations, not new LLM predictions.

They demonstrate how $P_o$ and $P_e$ affect κ mathematically.

However, they do not explain why the LLM changed its predictions under the actual manual-replacement conditions.

In the original study, both $P_o$ and $P_e$ changed across conditions.

This experiment does not determine which specific label transitions produced those changes or why they occurred.